# Problema
En este proyecto, exploramos un conjunto de datos que recopila diversas métricas de salud de pacientes cardíacos, como edad, presión arterial, frecuencia cardíaca, entre otras. Nuestro objetivo es desarrollar un modelo predictivo capaz de identificar con precisión a personas con enfermedades cardíacas. Dadas las graves implicaciones de no detectar un diagnóstico positivo, nuestro principal énfasis radica en garantizar que el modelo identifique a todos los pacientes potenciales, por lo que la exhaustividad (recall) para la clase positiva constituye una métrica crucial.

# Objetivos

* **Explorar el conjunto de datos:** Descubrir patrones, distribuciones y relaciones dentro de los datos.
* **Realizar un análisis exploratorio de datos (EDA) exhaustivo:** Analizar en profundidad las relaciones bivariadas respecto a la variable objetivo.

**Pasos de preprocesamiento:**
*  Eliminar características irrelevantes.
* Tratar los valores faltantes.
* Manejar los valores atípicos.
* Codificar las variables categóricas.
* Transformar las características con asimetría para aproximarlas a distribuciones normales.

**Construcción de modelos:**
* Establecer *pipelines* para aquellos modelos que requieran escalado.
* Implementar y ajustar modelos de clasificación, árboles de decisión y Random Forest.
* Priorizar un alto *recall* (exhaustividad) para la clase 1, garantizando una identificación exhaustiva de los pacientes cardíacos.


**Evaluar y comparar el rendimiento de los modelos:** Utilizar precisión (*precision*), exhaustividad (*recall*) y puntuación F1 (*F1-score*) para medir la efectividad de cada modelo.

# Importación de Librerías

In [1]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import ListedColormap
from sklearn.model_selection import train_test_split
from scipy.stats import boxcox
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import classification_report, accuracy_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

%matplotlib inline

In [2]:
# Establecer la resolución de las figuras generadas
plt.rcParams['figure.dpi'] = 200

# Configurar el estilo de Seaborn: fondo azul clínico claro con cuadrícula oscura
sns.set_theme(
    style='darkgrid',
    rc={
        'axes.facecolor': '#EBF2F7',   # Azul hielo/clínico muy suave para el área del gráfico
        'figure.facecolor': '#F4F8FA', # Tono casi blanco con matiz azul para el lienzo exterior
        'grid.color': '#D0DEE9'        # Líneas de cuadrícula en azul grisáceo tenue
    }
)

# Carga de Datos

In [5]:
# Leer dataset
df = pd.read_csv('https://raw.githubusercontent.com/david-ramirezl/data/refs/heads/main/heart.csv')
df.head()

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
0,52,1,0,125,212,0,1,168,0,1.0,2,2,3,0
1,53,1,0,140,203,1,0,155,1,3.1,0,0,3,0
2,70,1,0,145,174,0,1,125,1,2.6,0,0,3,0
3,61,1,0,148,203,0,1,161,0,0.0,2,1,3,0
4,62,0,0,138,294,1,1,106,0,1.9,1,3,2,0


# Conjunto de Datos

| Variable | Descripción |
| :--- | :--- |
| `age` | Edad del paciente en años. |
| `sex` | Sexo del paciente (0 = hombre, 1 = mujer). |
| `cp` | Tipo de dolor torácico:<br>• 0: Angina típica<br>• 1: Angina atípica<br>• 2: Dolor no anginoso<br>• 3: Asintomático |
| `trestbps` | Presión arterial en reposo (en mm Hg al ingresar al hospital). |
| `chol` | Colesterol sérico en mg/dl. |
| `fbs` | Nivel de glucosa en ayunas mayor a 120 mg/dl (1 = verdadero, 0 = falso). |
| `restecg` | Resultados electrocardiográficos en reposo:<br>• 0: Normal<br>• 1: Con anomalía de la onda ST-T<br>• 2: Con hipertrofia ventricular izquierda probable o definitiva |
| `thalach` | Frecuencia cardíaca máxima alcanzada durante la prueba de esfuerzo. |
| `exang` | Angina inducida por el ejercicio (1 = sí, 0 = no). |
| `oldpeak` | Depresión del ST inducida por el esfuerzo en relación con el reposo. |
| `slope` | Pendiente del segmento ST durante el pico de esfuerzo:<br>• 0: Ascendente<br>• 1: Plana<br>• 2: Descendente |
| `ca` | Número de vasos mayores (0-4) coloreados por fluoroscopía. |
| `thal` | Resultado de la prueba de esfuerzo con talio:<br>• 0: Normal<br>• 1: Defecto fijo<br>• 2: Defecto reversible<br>• 3: No descrito |
| `target` | Presencia de enfermedad cardíaca (0 = sin enfermedad, 1 = presencia de enfermedad). |

# Descripción General del Conjunto

In [4]:
# Mostrar un resumen conciso del DataFrame
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1025 entries, 0 to 1024
Data columns (total 14 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       1025 non-null   int64  
 1   sex       1025 non-null   int64  
 2   cp        1025 non-null   int64  
 3   trestbps  1025 non-null   int64  
 4   chol      1025 non-null   int64  
 5   fbs       1025 non-null   int64  
 6   restecg   1025 non-null   int64  
 7   thalach   1025 non-null   int64  
 8   exang     1025 non-null   int64  
 9   oldpeak   1025 non-null   float64
 10  slope     1025 non-null   int64  
 11  ca        1025 non-null   int64  
 12  thal      1025 non-null   int64  
 13  target    1025 non-null   int64  
dtypes: float64(1), int64(13)
memory usage: 112.2 KB


### Observaciones:
- **Número de registros:** El conjunto de datos contiene 1025 entradas, con índices del 0 al 1024.
- **Columnas:** Existen 14 columnas que corresponden a diversos atributos de los pacientes y resultados de pruebas clínicas.
- **Tipos de datos:**
  - La mayoría de las columnas (13 de 14) corresponden al tipo de dato `int64`.
  - Solo la columna `oldpeak` es de tipo `float64`.
- **Valores nulos:** No se observan valores faltantes en el conjunto de datos, ya que cada columna cuenta con 1025 entradas no nulas (*non-null*).

> **Nota:** Según los tipos de datos y las explicaciones de las características revisadas anteriormente, se observa que 9 columnas (`sex`, `cp`, `fbs`, `restecg`, `exang`, `slope`, `ca`, `thal` y `target`) son numéricas a nivel de tipo de dato, pero categóricas en cuanto a su semántica. Estas variables deben convertirse al tipo de dato cadena (`object` o `str`) para permitir un análisis y una interpretación adecuados:

In [6]:
# Definir las características continuas
caracteristicas_continuas = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']

# Identificar las características que se deben convertir al tipo de dato object
caracteristicas_a_convertir = [columna for columna in df.columns if columna not in caracteristicas_continuas]

# Convertir las características identificadas al tipo de dato object
df[caracteristicas_a_convertir] = df[caracteristicas_a_convertir].astype('object')

df.dtypes

age           int64
sex          object
cp           object
trestbps      int64
chol          int64
fbs          object
restecg      object
thalach       int64
exang        object
oldpeak     float64
slope        object
ca           object
thal         object
target       object
dtype: object

# Estadísticas descriptivas de las variables numéricas

Analicemos las estadísticas descriptivas de nuestras variables numéricas:

In [7]:
# Obtener las estadísticas descriptivas de las variables numéricas
df.describe().T

,count,mean,std,min,25%,50%,75%,max
age,1025.0,54.434146,9.072290,29.0,48.0,56.0,61.0,77.0
trestbps,1025.0,131.611707,17.516718,94.0,120.0,130.0,140.0,200.0
chol,1025.0,246.000000,51.592510,126.0,211.0,240.0,275.0,564.0
thalach,1025.0,149.114146,23.005724,71.0,132.0,152.0,166.0,202.0
oldpeak,1025.0,1.071512,1.175053,0.0,0.0,0.8,1.8,6.2


### Variables numéricas:
- **`age`:** La edad promedio de los pacientes es de aproximadamente 54.4 años, con un mínimo de 29 y un máximo de 77 años.
- **`trestbps`:** La presión arterial en reposo promedio es de cerca de 131.61 mm Hg, con un rango que va de 94 a 200 mm Hg.
- **`chol`:** El nivel promedio de colesterol es de aproximadamente 246 mg/dl, con un mínimo de 126 y un máximo de 564 mg/dl.
- **`thalach`:** La frecuencia cardíaca máxima alcanzada promedio se sitúa alrededor de 149.11 lpm, en un rango de 71 a 202.
- **`oldpeak`:** La depresión del ST promedio inducida por el ejercicio en relación con el reposo es de cerca de 1.07, con valores que oscilan entre 0 y 6.2.

# Estadísticas descriptivas de las variables categóricas

A continuación, revisemos las estadísticas descriptivas de las características categóricas: